<a href="https://colab.research.google.com/github/nainaakasaudhan25/CommunityClassroom-Git/blob/master/finalyrproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers
!pip install torch
!pip install scikit-learn
!pip install pandas

In [2]:
from google.colab import files
uploaded = files.upload()

Saving ai_mock_interview_labeled_dataset_30000_rows.csv to ai_mock_interview_labeled_dataset_30000_rows.csv


In [21]:
import pandas as pd
df = pd.read_csv("ai_mock_interview_labeled_dataset_30000_rows.csv")

In [22]:
df["text"] = "Question: " + df["question"] + " Answer: " + df["answer"]

In [23]:
df = df.drop_duplicates(subset=["text"])
print(df.shape)

(1560, 6)


In [ ]:
#from sklearn.model_selection import train_test_split

#train_texts, val_texts, train_labels, val_labels = train_test_split(
 #  df["label"].tolist(),
  #  test_size=0.2,
   # random_state=42,
    #stratify=df["label"]
#)

In [24]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx, val_idx = next(
    gss.split(df["text"], df["label"], groups=df["question_id"])
)

In [25]:
train_df = df.iloc[train_idx]
val_df = df.iloc[val_idx]

In [26]:
train_texts = train_df["text"].tolist()
train_labels = train_df["label"].tolist()

val_texts = val_df["text"].tolist()
val_labels = val_df["label"].tolist()

In [27]:
#Check data leakage
print(len(set(train_texts).intersection(set(val_texts))))

0


In [28]:
print(len(train_texts))
print(len(val_texts))
print(len(train_labels))
print(len(val_labels))

1248
312
1248
312


In [29]:
from transformers import BertTokenizer
tokenizer=BertTokenizer.from_pretrained("bert-base-uncased")

In [30]:
#tokenize training Data
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=256
)

In [31]:
#tokenize validation data
val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding=True,
    max_length=256
)

In [32]:
import torch

class InterviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)



In [33]:
train_dataset = InterviewDataset(train_encodings, train_labels)
val_dataset = InterviewDataset(val_encodings, val_labels)

In [34]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [35]:
import torch
torch.cuda.is_available()

True

In [41]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_steps=100,
    save_total_limit=2
)

In [42]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):

    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted'
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [43]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [44]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


{'eval_loss': 1.1740704774856567,
 'eval_model_preparation_time': 0.0028,
 'eval_accuracy': 0.3333333333333333,
 'eval_f1': 0.16666666666666666,
 'eval_precision': 0.1111111111111111,
 'eval_recall': 0.3333333333333333,
 'eval_runtime': 1.5963,
 'eval_samples_per_second': 195.456,
 'eval_steps_per_second': 24.432}

In [45]:
trainer.train()

Step,Training Loss
100,0.083348
200,0.000770
300,0.000394
400,0.000276
500,0.000224
600,0.000196


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=624, training_loss=0.013662360322016936, metrics={'train_runtime': 114.949, 'train_samples_per_second': 43.428, 'train_steps_per_second': 5.428, 'total_flos': 179574907599360.0, 'train_loss': 0.013662360322016936, 'epoch': 4.0})

In [46]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.00013568073336500674, 'eval_model_preparation_time': 0.0028, 'eval_accuracy': 1.0, 'eval_f1': 1.0, 'eval_precision': 1.0, 'eval_recall': 1.0, 'eval_runtime': 1.5701, 'eval_samples_per_second': 198.712, 'eval_steps_per_second': 24.839, 'epoch': 4.0}


In [47]:
preds = trainer.predict(val_dataset)

y_true = preds.label_ids
y_pred = preds.predictions.argmax(-1)

In [48]:
from sklearn.metrics import classification_report, confusion_matrix

print("Classification Report:\n", classification_report(y_true, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       104
           1       1.00      1.00      1.00       104
           2       1.00      1.00      1.00       104

    accuracy                           1.00       312
   macro avg       1.00      1.00      1.00       312
weighted avg       1.00      1.00      1.00       312

Confusion Matrix:
 [[104   0   0]
 [  0 104   0]
 [  0   0 104]]


In [49]:
import random

def shuffle_words(text):
    words = text.split()
    random.shuffle(words)
    return " ".join(words)

In [50]:
shuffled_val_texts = [shuffle_words(t) for t in val_texts]

In [51]:
shuffled_encodings = tokenizer(
    shuffled_val_texts,
    truncation=True,
    padding=True,
    max_length=256
)

In [52]:
shuffled_dataset = InterviewDataset(shuffled_encodings, val_labels)

In [57]:
preds = trainer.predict(shuffled_dataset)
y_pred = preds.predictions.argmax(-1)

In [58]:
from sklearn.metrics import accuracy_score

print("Accuracy on shuffled text:", accuracy_score(val_labels, y_pred))

Accuracy on shuffled text: 0.6089743589743589


In [59]:
model.save_pretrained("mock_interview_model")
tokenizer.save_pretrained("mock_interview_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('mock_interview_model/tokenizer_config.json',
 'mock_interview_model/tokenizer.json')

In [63]:
import torch

def evaluate_answer(question, answer):

    text = "Question: " + question + " Answer: " + answer

    # tokenize input
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    # move inputs to same device as model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # inference
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.argmax(outputs.logits, dim=1).item()

    return pred

In [64]:
question = "Explain machine learning"

answer1 = "Machine learning allows computers to learn patterns from data and make predictions."

answer2 = "I don't know anything about this."

print(evaluate_answer(question, answer1))
print(evaluate_answer(question, answer2))

1
0


In [65]:
def generate_feedback(score):

    if score == 0:
        return "Your answer is weak. Try explaining the concept clearly."

    elif score == 1:
        return "Your answer is okay but needs more details or examples."

    else:
        return "Excellent answer with clear explanation."

In [66]:
score = evaluate_answer(question, answer1)

print("Score:", score)
print("Feedback:", generate_feedback(score))

Score: 1
Feedback: Your answer is okay but needs more details or examples.
